# Create crop fine-tuning data
Reuse `charSimilarityMLP/training_data.pt` as the authoritative accepted GT labels,
sequence split, and candidate validation/test pairs. No GT assignment is rerun, and predicted
cross-page identities never become labels. Run the existing data notebook first if
that source file needs rebuilding.

All appearances remain available for unchanged clustering; only accepted appearances
can enter triplets or supervised pairs. Cached `raw_embedding` is used for mining
only and is never passed to the trainable encoder. This notebook saves one file.

Exclude `NARRATION` and `UNKNOWN` (ignoring case and surrounding spaces) from all
triplet/pair supervision. Their crops still participate in clustering. Run this
notebook, then training, then testing to replace the previous experiment artifacts.


In [4]:
import torch
import torch.nn.functional as F
import numpy as np
import hashlib, random
from pathlib import Path
from PIL import Image

ROOT = Path('C:/Users/ESHAAN/BYTE/Applied-Task/Magi/fineTuningMagiCharSimHeads')
DATA_FILE = ROOT / 'training_data.pt'
MODEL_FILE = ROOT / 'magi_crop_last_block.pt'
BASE_MODEL = Path('C:/MODELS/magiv2')

def file_hash(path):
    with Path(path).open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()

SOURCE_FILE = ROOT.parent / 'charSimilarityMLP' / 'training_data.pt'
SEED = 42
HARD_NEGATIVE_RATIO, HARD_NEGATIVE_TOP_K = 0.70, 20
MAX_POSITIVES_PER_SEQUENCE = 4000
MAX_SEQUENCE_SHARE = 0.25  # Stop if one sequence supplies more than 25% of training triplets.
EXCLUDED_IDENTITIES = {'NARRATION', 'UNKNOWN'}
IMAGES = ROOT.parent / 'TaskData' / 'development' / 'images'

source_hash = file_hash(SOURCE_FILE)
source = torch.load(SOURCE_FILE, map_location='cpu', weights_only=True)
assert source['version'] == 2
splits = source['splits']
sequences, excluded_counts = [], {'NARRATION': 0, 'UNKNOWN': 0}
for original in source['sequences']:
    sequence = dict(original)
    sequence['gt'] = original['gt'].clone()
    # Keep every crop for clustering, but remove non-character labels from supervision.
    for row in torch.where(sequence['gt'] >= 0)[0].tolist():
        name = sequence['speaker_names'][int(sequence['gt'][row])].strip().upper()
        if name in EXCLUDED_IDENTITIES:
            sequence['gt'][row] = -1
            excluded_counts[name] += 1
    sequences.append(sequence)
print('Excluded supervised appearances:', excluded_counts)
split_sets = [set(splits[name]) for name in ('train', 'validation', 'test')]
assert all(split_sets) and all(not a & b for i, a in enumerate(split_sets) for b in split_sets[i+1:])
assert set.union(*split_sets) == set(range(len(sequences)))

appearances, sequence_rows = [], {}
for i, sequence in enumerate(sequences):
    sequence_rows[i] = []
    for row in range(len(sequence['gt'])):
        page, char = int(sequence['page'][row]), int(sequence['character'][row])
        gt = int(sequence['gt'][row])
        path = IMAGES / sequence['id'] / sequence['page_files'][page]
        assert path.is_file(), f'Missing page: {path}'
        sequence_rows[i].append(len(appearances))
        appearances.append(dict(sequence_idx=i, sequence_id=sequence['id'], sequence_row=row,
            page_idx=page, page_path=str(path), char_idx=char,
            appearance_id=f'PG{page+1}:C{char+1}', bbox=sequence['boxes'][row].tolist(),
            gt_identity=sequence['speaker_names'][gt] if gt >= 0 else None,
            raw_embedding=sequence['embeddings'][row]))
print('Reused source dataset:', SOURCE_FILE)


Excluded supervised appearances: {'NARRATION': 29, 'UNKNOWN': 14}
Reused source dataset: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\charSimilarityMLP\training_data.pt


## Fixed triplets and held-out pairs
Prefer cross-page positives, using same-page positives only when that identity has
no cross-page positive. Each unordered anchor-positive combination occurs once.
Negatives come from another accepted identity in the same sequence. Choose 70%
from the top 20 raw-cosine candidates and 30% at random (rounded per sequence).
Validation triplets are fixed diagnostics. Retain original validation/test pairs
only when both endpoints still have an accepted character identity.


In [5]:
from itertools import combinations
from collections import Counter

def make_triplets(indices, seed):
    triplets = []
    for i in indices:
        sequence = sequences[i]
        rng = random.Random(f'{seed}:{sequence["id"]}')
        accepted = torch.where(sequence['gt'] >= 0)[0].tolist()
        groups = {}
        for row in accepted:
            groups.setdefault(int(sequence['gt'][row]), []).append(row)
        positives = []
        for members in groups.values():
            pairs = list(combinations(members, 2))
            cross = [(a, b) for a, b in pairs if sequence['page'][a] != sequence['page'][b]]
            positives.extend(cross or pairs)
        rng.shuffle(positives)
        positives.sort(key=lambda pair: int(sequence['page'][pair[0]]) == int(sequence['page'][pair[1]]))
        positives = positives[:MAX_POSITIVES_PER_SEQUENCE] if len(groups) >= 2 else []
        kinds = ['hard'] * round(len(positives) * HARD_NEGATIVE_RATIO)
        kinds += ['random'] * (len(positives) - len(kinds))
        rng.shuffle(kinds)
        unit = F.normalize(sequence['embeddings'], dim=-1)
        for (a, p), kind in zip(positives, kinds):
            if rng.random() < 0.5:
                a, p = p, a
            candidates = [n for n in accepted if sequence['gt'][n] != sequence['gt'][a]]
            if kind == 'hard':
                candidates.sort(key=lambda n: float(unit[a] @ unit[n]), reverse=True)
                candidates = candidates[:HARD_NEGATIVE_TOP_K]
            n = rng.choice(candidates)
            triplets.append(dict(anchor_idx=sequence_rows[i][a], positive_idx=sequence_rows[i][p],
                negative_idx=sequence_rows[i][n], negative_type=kind,
                raw_pos_similarity=float(unit[a] @ unit[p]),
                raw_neg_similarity=float(unit[a] @ unit[n])))
    return triplets

def make_pairs(split):
    pairs = []
    for i, a, b, label in source['pairs'][split].tolist():
        sequence = sequences[i]
        assert i in splits[split] and a != b
        if sequence['gt'][a] < 0 or sequence['gt'][b] < 0:
            continue
        assert label == int(sequence['gt'][a] == sequence['gt'][b])
        pairs.append(dict(a_idx=sequence_rows[i][a], b_idx=sequence_rows[i][b], label=label,
                          cross_page=bool(sequence['page'][a] != sequence['page'][b])))
    assert {pair['label'] for pair in pairs if pair['cross_page']} == {0, 1}
    return pairs

train_triplets = make_triplets(splits['train'], SEED)
val_triplets = make_triplets(splits['validation'], SEED + 1)
val_pairs, test_pairs = make_pairs('validation'), make_pairs('test')
assert train_triplets and val_triplets, 'Need usable training and validation triplets.'
for split, triplets in [('train', train_triplets), ('validation', val_triplets)]:
    seen = set()
    for triplet in triplets:
        a, p, n = [appearances[triplet[key]] for key in ('anchor_idx', 'positive_idx', 'negative_idx')]
        assert a['sequence_idx'] == p['sequence_idx'] == n['sequence_idx']
        assert a['sequence_idx'] in splits[split]
        assert a['gt_identity'] == p['gt_identity'] and a['gt_identity'] != n['gt_identity']
        assert all(record['gt_identity'] is not None and
                   record['gt_identity'].strip().upper() not in EXCLUDED_IDENTITIES for record in (a, p, n))
        key = tuple(sorted((triplet['anchor_idx'], triplet['positive_idx'])))
        assert key not in seen
        seen.add(key)

# Print every sequence, including those that cannot supply a triplet.
triplet_counts = {}
for split, triplets in [('train', train_triplets), ('validation', val_triplets)]:
    counts = Counter(appearances[t['anchor_idx']]['sequence_idx'] for t in triplets)
    triplet_counts[split] = dict(counts)
    print(f'\n{split} triplets per sequence:')
    for i in splits[split]:
        print(sequences[i]['id'], '| triplets:', counts[i], '| share:', f'{counts[i] / len(triplets):.1%}')
    largest = max(counts, key=counts.get)
    share = counts[largest] / len(triplets)
    print('Largest sequence:', sequences[largest]['id'], '| share:', f'{share:.1%}')
    if split == 'train':
        assert share <= MAX_SEQUENCE_SHARE, (
            'One sequence dominates training. Lower MAX_POSITIVES_PER_SEQUENCE and rebuild before training.')

# Pair tensors use the same filtered endpoints as pair dictionaries and clustering evaluation.
filtered_pairs = {}
for split, pairs in [('validation', val_pairs), ('test', test_pairs)]:
    rows = []
    for pair in pairs:
        a, b = appearances[pair['a_idx']], appearances[pair['b_idx']]
        assert all(record['gt_identity'] is not None and
                   record['gt_identity'].strip().upper() not in EXCLUDED_IDENTITIES for record in (a, b))
        rows.append([a['sequence_idx'], a['sequence_row'], b['sequence_row'], pair['label']])
    filtered_pairs[split] = torch.tensor(rows, dtype=torch.long).reshape(-1, 4)
data = dict(version=2, appearances=appearances, sequence_rows=sequence_rows,
    sequences=sequences, splits=splits, pairs=filtered_pairs,
    train_triplets=train_triplets, val_triplets=val_triplets, val_pairs=val_pairs, test_pairs=test_pairs,
    source_dataset_hash=source_hash, triplet_counts=triplet_counts, settings=dict(seed=SEED, hard_negative_ratio=HARD_NEGATIVE_RATIO,
        hard_negative_top_k=HARD_NEGATIVE_TOP_K, prefer_cross_page_positive=True,
        max_positives_per_sequence=MAX_POSITIVES_PER_SEQUENCE,
        excluded_identities=sorted(EXCLUDED_IDENTITIES), max_sequence_share=MAX_SEQUENCE_SHARE))
assert file_hash(SOURCE_FILE) == source_hash, 'Source changed while building data.'
torch.save(data, DATA_FILE)
print('Dataset SHA256:', file_hash(DATA_FILE))



train triplets per sequence:
seq_b8b5563330fdc89d | triplets: 22 | share: 2.9%
seq_e8b4f75454f9491f | triplets: 14 | share: 1.8%
seq_5f9ee35846b7e635 | triplets: 8 | share: 1.0%
seq_b9b5a9ad9a64ced9 | triplets: 14 | share: 1.8%
seq_84541a1f4b159018 | triplets: 37 | share: 4.8%
seq_d749b853cf795b6a | triplets: 15 | share: 2.0%
seq_2032620aa4e4ac7f | triplets: 4 | share: 0.5%
seq_b136f32c2233b97c | triplets: 0 | share: 0.0%
seq_d8afceb0e8f9122d | triplets: 8 | share: 1.0%
seq_96839890d69d7698 | triplets: 6 | share: 0.8%
seq_45c9871f52e99417 | triplets: 22 | share: 2.9%
seq_f7f9d9dbf5b05cb5 | triplets: 29 | share: 3.8%
seq_da71f7a842c91090 | triplets: 21 | share: 2.7%
seq_952f154fb1505883 | triplets: 10 | share: 1.3%
seq_a3bd36201a0cd61b | triplets: 0 | share: 0.0%
seq_de542b63e6b19bc2 | triplets: 2 | share: 0.3%
seq_71033fec4de494fc | triplets: 28 | share: 3.7%
seq_d7af6c64ebc22bbe | triplets: 3 | share: 0.4%
seq_fa2eee6269a5dad3 | triplets: 22 | share: 2.9%
seq_06ec0acde37dffa4 | tripl

In [6]:
print('Accepted appearances:', sum(a['gt_identity'] is not None for a in appearances))
for name, indices in splits.items():
    print(name, 'sequences:', len(indices))
print('Train triplets:', len(train_triplets))
print('Cross-page positives:', sum(appearances[t['anchor_idx']]['page_idx'] !=
    appearances[t['positive_idx']]['page_idx'] for t in train_triplets))
print('Negative types:', dict(Counter(t['negative_type'] for t in train_triplets)))
print('Validation cross-page pairs:', sum(p['cross_page'] for p in val_pairs))
print('Test cross-page pairs:', sum(p['cross_page'] for p in test_pairs))
for triplet in train_triplets[:5]:
    print()
    for label, key in [('Anchor', 'anchor_idx'), ('Positive', 'positive_idx'), ('Negative', 'negative_idx')]:
        a = appearances[triplet[key]]
        print(label, a['sequence_id'], a['appearance_id'], a['gt_identity'])
    print('Raw AP:', round(triplet['raw_pos_similarity'], 3), '| raw AN:',
          round(triplet['raw_neg_similarity'], 3), '|', triplet['negative_type'])
print('Saved:', DATA_FILE)


Accepted appearances: 827
train sequences: 56
validation sequences: 12
test sequences: 12
Train triplets: 765
Cross-page positives: 700
Negative types: {'hard': 539, 'random': 226}
Validation cross-page pairs: 405
Test cross-page pairs: 793

Anchor seq_b8b5563330fdc89d PG3:C6 character3
Positive seq_b8b5563330fdc89d PG2:C13 character3
Negative seq_b8b5563330fdc89d PG1:C11 character2
Raw AP: 0.732 | raw AN: 0.859 | hard

Anchor seq_b8b5563330fdc89d PG1:C6 character2
Positive seq_b8b5563330fdc89d PG3:C5 character2
Negative seq_b8b5563330fdc89d PG3:C9 character4
Raw AP: 0.376 | raw AN: 0.453 | hard

Anchor seq_b8b5563330fdc89d PG1:C11 character2
Positive seq_b8b5563330fdc89d PG2:C7 character2
Negative seq_b8b5563330fdc89d PG3:C6 character3
Raw AP: 0.955 | raw AN: 0.859 | hard

Anchor seq_b8b5563330fdc89d PG2:C16 character2
Positive seq_b8b5563330fdc89d PG1:C11 character2
Negative seq_b8b5563330fdc89d PG3:C6 character3
Raw AP: 0.859 | raw AN: 0.808 | random

Anchor seq_b8b5563330fdc89d PG3